In [ ]:
# 1. 라이브러리 및 데이터 로드 (Phase 18: V2 엔진 롤백 및 트리 구속 해제)
import pandas as pd
import numpy as np
import os
import joblib
import json
import shutil
import warnings
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

DATA_DIR = r"C:\Users\이호준\OneDrive\바탕 화면\LG aimers\open\data"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "./data"

ID_COL = "row_id"
TARGET_COL = "control_success"
CAT_COLS = ["top_bottom", "game_type", "base_state", "pitcher_hand", "batter_hand", "count_state"]

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), encoding="utf-8-sig")
test_cols = pd.read_csv(os.path.join(DATA_DIR, "test.csv"), encoding="utf-8-sig", nrows=0).columns

train['count_state'] = train['balls_before'].astype(str) + "-" + train['strikes_before'].astype(str)

# 🔥 결측치(NaN) 이슈가 없었던 가장 완벽한 V2 물리 엔진으로 롤백합니다!
tm_stats = pd.read_csv(os.path.join(DATA_DIR, "pitcher_trackman_stats_v2.csv"))
train = pd.merge(train, tm_stats, on='pitcher_id', how='left')

tm_features = ['tm_rel_speed', 'tm_spin_rate', 'tm_ivb', 'tm_hb', 'tm_extension', 
               'tm_rel_height', 'tm_rel_side', 'tm_rel_speed_std', 'tm_hb_std', 'tm_ivb_std']
FEATURES = [c for c in test_cols if c != ID_COL] + ['count_state'] + tm_features
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]
print(f"결측치 없는 V2 데이터 로드 완료! (총 피처 수: {len(FEATURES)})")


In [ ]:
# 2. 전처리 파이프라인
preprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_COLS),
    ("num", SimpleImputer(strategy="median"), NUM_COLS),
])


In [ ]:
# 3. 시간 누수 방지 2024년 정밀 Isotonic 캘리브레이션 (구속 해제)
is_val = train["season"] == 2024
X_train, y_train = train.loc[~is_val, FEATURES], train.loc[~is_val, TARGET_COL].values
X_val, y_val = train.loc[is_val, FEATURES], train.loc[is_val, TARGET_COL].values

print("전처리 적용 중...")
preprocessor.fit(X_train)
X_train_pre = preprocessor.transform(X_train)
X_val_pre = preprocessor.transform(X_val)

SEEDS = [42, 43, 44, 45, 46]
ensemble_preds = np.zeros(len(X_val))

print("🔥 족쇄가 풀린 15개 시드 앙상블 학습 중... (min_child 500 -> 100, trees 300 -> 500)")
for s in SEEDS:
    lgb_m = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31, max_depth=6, min_child_samples=100, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=s, verbosity=-1)
    lgb_m.fit(X_train_pre, y_train)
    p_lgb = lgb_m.predict_proba(X_val_pre)[:, 1]
    
    xgb_m = xgb.XGBClassifier(n_estimators=500, learning_rate=0.03, max_depth=6, min_child_weight=100, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=s, tree_method='hist')
    xgb_m.fit(X_train_pre, y_train)
    p_xgb = xgb_m.predict_proba(X_val_pre)[:, 1]
    
    cb_m = cb.CatBoostClassifier(iterations=500, learning_rate=0.03, depth=6, min_data_in_leaf=100, random_seed=s, verbose=0)
    cb_m.fit(X_train_pre, y_train)
    p_cb = cb_m.predict_proba(X_val_pre)[:, 1]
    
    ensemble_preds += (p_lgb + p_xgb + p_cb) / 3.0

ensemble_preds /= len(SEEDS)

print("\n🔧 2024년 퓨어 정통 Isotonic 보정기 가동 중...")
iso = IsotonicRegression(out_of_bounds='clip')
calibrated_preds = iso.fit_transform(ensemble_preds, y_val)

r = y_val.mean()
brier = ((calibrated_preds - y_val) ** 2).mean()
baseline_brier = r * (1 - r)
score = max(0, 100000 * (1 - brier / baseline_brier))

print(f"\n✅ Brier Score: {brier:.6f} | 기준선 r(1-r): {baseline_brier:.6f}")
print(f"🚀 족쇄 해제 V2 Validation Score: {score:.2f} (900점 폭파 대기 중!)")

iso_data = {
    "X_thresh": iso.X_thresholds_.tolist(),
    "y_thresh": iso.y_thresholds_.tolist()
}
os.makedirs("./submission/model", exist_ok=True)
with open("./submission/model/iso_thresholds.json", "w") as f:
    json.dump(iso_data, f)


In [ ]:
# 4. 전체 데이터 재학습 및 에러 없는 script.py 생성
import zipfile

print("전체 데이터 전처리 및 구속 해제 15시드 모델 재학습 중...")
preprocessor.fit(train[FEATURES])
joblib.dump(preprocessor, "./submission/model/preprocessor.pkl", compress=3)
joblib.dump(FEATURES, "./submission/model/features.pkl")
shutil.copy(os.path.join(DATA_DIR, "pitcher_trackman_stats_v2.csv"), "./submission/model/pitcher_trackman_stats.csv")

X_all_pre = preprocessor.transform(train[FEATURES])
y_all = train[TARGET_COL].values

for i, s in enumerate(SEEDS):
    lgb_m = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31, max_depth=6, min_child_samples=100, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=s, verbosity=-1)
    lgb_m.fit(X_all_pre, y_all)
    lgb_m.booster_.save_model(f"./submission/model/lgb_seed{i}.txt")
    
    xgb_m = xgb.XGBClassifier(n_estimators=500, learning_rate=0.03, max_depth=6, min_child_weight=100, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=s, tree_method='hist')
    xgb_m.fit(X_all_pre, y_all)
    xgb_m.save_model(f"./submission/model/xgb_seed{i}.json")
    
    cb_m = cb.CatBoostClassifier(iterations=500, learning_rate=0.03, depth=6, min_data_in_leaf=100, random_seed=s, verbose=0)
    cb_m.fit(X_all_pre, y_all)
    cb_m.save_model(f"./submission/model/cb_seed{i}.cbm")

print("15개 구속 해제 모델 학습 및 저장 완료.")

with open("./submission/requirements.txt", "w") as f:
    f.write("pandas\nscikit-learn\nlightgbm\nxgboost\ncatboost\njoblib\n")

script_code = """import os
import joblib
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

ID_COL = "row_id"
TARGET_COL = "control_success"
N_SEEDS = 5

def merge_predictions(sub, ids, preds):
    pred_map = dict(zip(ids, preds))
    values = [pred_map.get(rid, cur) for rid, cur in zip(sub[ID_COL], sub[TARGET_COL])]
    sub[TARGET_COL] = values
    return sub

def main():
    TEST_PATH = "./data/test.csv"
    SAMPLE_SUB_PATH = "./data/sample_submission.csv"
    OUT_PATH = "./output/submission.csv"

    preprocessor = joblib.load("./model/preprocessor.pkl")
    FEATURES = joblib.load("./model/features.pkl")
    tm_stats = pd.read_csv("./model/pitcher_trackman_stats.csv")
    
    with open("./model/iso_thresholds.json", "r") as f:
        iso_data = json.load(f)
    X_thresh = np.array(iso_data["X_thresh"])
    y_thresh = np.array(iso_data["y_thresh"])
    
    test = pd.read_csv(TEST_PATH, encoding="utf-8-sig")
    sub = pd.read_csv(SAMPLE_SUB_PATH, encoding="utf-8-sig")
    
    test['count_state'] = test['balls_before'].astype(str) + "-" + test['strikes_before'].astype(str)
    test = pd.merge(test, tm_stats, on='pitcher_id', how='left')
    
    ids = test[ID_COL].tolist()
    X_pre = preprocessor.transform(test[FEATURES])
    
    final_preds = np.zeros(len(test))
    
    for i in range(N_SEEDS):
        lgb_booster = lgb.Booster(model_file=f"./model/lgb_seed{i}.txt")
        xgb_model = xgb.XGBClassifier()
        xgb_model.load_model(f"./model/xgb_seed{i}.json")
        cb_model = cb.CatBoostClassifier()
        cb_model.load_model(f"./model/cb_seed{i}.cbm")
        
        p_lgb = lgb_booster.predict(X_pre)
        p_xgb = xgb_model.predict_proba(X_pre)[:, 1]
        p_cb = cb_model.predict_proba(X_pre)[:, 1]
        
        final_preds += (p_lgb + p_xgb + p_cb) / 3.0
        
    final_preds /= N_SEEDS
    
    calibrated_preds = np.interp(final_preds, X_thresh, y_thresh)
    
    sub = merge_predictions(sub, ids, calibrated_preds)
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    sub.to_csv(OUT_PATH, index=False, encoding="utf-8")

if __name__ == "__main__":
    main()
"""

with open("./submission/script.py", "w", encoding="utf-8") as f:
    f.write(script_code)

zip_path = 'unchained_v2_submit.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk('./submission'):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, os.path.relpath(file_path, './submission'))

print("✅ 900점 조준 완료! 족쇄가 풀린 인공지능 unchained_v2_submit.zip 생성 완료!")
